In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# EV Battery Degradation Analysis: NMC vs. LFP Chemistries

## 1. Project structure and Introduction

### 1.1 The EV Battery Landscape

Currently, the market is dominated by two lithium-ion battery chemistries: **Nickel Manganese Cobalt (NMC)** and **Lithium Iron Phosphate (LFP)**.

### Battery Chemistry Comparison:

* **NMC (Nickel Manganese Cobalt):**
    * **Pros:** High energy density (longer driving range), better performance in cold climates.
    * **Cons:** Shorter cycle life, higher cost, lower thermal stability.
* **LFP (Lithium Iron Phosphate):**
    * **Pros:** Exceptional longevity (more charge cycles), lower cost, superior thermal stability (safety).
    * **Cons:** Lower energy density (heavier for the same range), slower performance in extreme cold.

### 1.2 Project Overview
This project aims to compare the degradation behavior of NMC and LFP battery chemistries using data science workflows. By analyzing factors such as charging speed, temperature, and usage cycles, we seek to visualize how these two technologies age differently over time.

### 1.3 Objectives
* **Comparative Analysis:** Evaluate the State of Health (SoH) and capacity retention of NMC and LFP batteries.
* **Factor Impact:** Analyze how external factors (temperature, discharge rates, fast-charging) affect degradation in both chemistries.
* **Predictive Insights:** Identify patterns in internal resistance growth and capacity loss.

### 1.4 Data Sources and Methodology
The project utilizes two independent synthetic datasets to provide a multi-layered view of battery health:
1.  **Vehicle-Level Data ([EV Battery Degradation and Charge](https://www.kaggle.com/datasets/bertnardomariouskono/electric-vehicle-ev-battery-degradation-and-charge)):** Focuses on high-level vehicle metrics like age, driving style, and fast-charging ratios across different car models.
2.  **Cycle-Level Data ([Battery Failure Surfaces](https://www.kaggle.com/datasets/niladriroy0/battery-failure-surfaces)):** Provides granular, cycle-by-cycle measurements of internal resistance and capacity retention, simulating laboratory testing conditions.

### 1.5 Important Disclaimer & Limitations
**Note:** This analysis is based on **synthetic datasets** generated for educational and research simulation purposes. This project is a comparison of two specific synthetic data sources and should **not** be interpreted as a factual, real-world verdict on the ultimate performance of NMC or LFP chemistries. The conclusions reached here are representative of the trends within these specific datasets and are intended to demonstrate data science methodologies rather than provide a definitive engineering benchmark.

## 2. Data Acquisition and Initial Processing
In this section, we load our two independent datasets, inspect their structures, and perform initial cleaning to ensure consistent naming conventions for our battery chemistry comparison.

In [15]:
# Loading datasets
vehicle_data = pd.read_csv('data/ev_battery_degradation_v1.csv')
cycle_data = pd.read_csv('data/battery_failure_surfaces.csv')

In [16]:
vehicle_data.info()
display(vehicle_data.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Vehicle_ID               10000 non-null  object 
 1   Car_Model                10000 non-null  object 
 2   Battery_Type             10000 non-null  object 
 3   Battery_Capacity_kWh     10000 non-null  float64
 4   Vehicle_Age_Months       10000 non-null  int64  
 5   Total_Charging_Cycles    10000 non-null  int64  
 6   Avg_Temperature_C        10000 non-null  float64
 7   Fast_Charge_Ratio        10000 non-null  float64
 8   Avg_Discharge_Rate_C     10000 non-null  float64
 9   Driving_Style            10000 non-null  object 
 10  Internal_Resistance_Ohm  10000 non-null  float64
 11  SoH_Percent              10000 non-null  float64
 12  Battery_Status           10000 non-null  object 
dtypes: float64(6), int64(2), object(5)
memory usage: 1015.8+ KB


,Vehicle_ID,Car_Model,Battery_Type,Battery_Capacity_kWh,Vehicle_Age_Months,Total_Charging_Cycles,Avg_Temperature_C,Fast_Charge_Ratio,Avg_Discharge_Rate_C,Driving_Style,Internal_Resistance_Ohm,SoH_Percent,Battery_Status
0,1fb46ae8,Tesla Model 3,NMC,75.0,41,390,21.5,0.51,2.22,Aggressive,0.0362,94.60,Healthy
1,b7ef35aa,Tesla Model 3,NMC,75.0,29,401,18.0,0.62,1.34,Aggressive,0.0333,95.68,Healthy
2,76cb49e0,Ford Mustang Mach-E,NMC,88.0,71,941,18.4,0.78,1.48,Conservative,0.0526,89.80,Healthy
3,456a7aef,Ford Mustang Mach-E,NMC,88.0,57,378,10.8,0.61,0.72,Moderate,0.0314,96.29,Healthy
4,bd758049,Tesla Model 3,NMC,75.0,58,239,30.3,0.89,1.48,Conservative,0.0297,96.75,Healthy


In [17]:
cycle_data.info()
display(cycle_data.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 477253 entries, 0 to 477252
Data columns (total 11 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   cell_chemistry               477253 non-null  object 
 1   cycle                        477253 non-null  int64  
 2   charge_rate_C                477253 non-null  float64
 3   discharge_rate_C             477253 non-null  float64
 4   cell_temperature_C           477253 non-null  float64
 5   internal_resistance_mOhm     477253 non-null  float64
 6   capacity_retention_%         477253 non-null  float64
 7   cumulative_high_temp_cycles  477253 non-null  int64  
 8   fast_charge_exposure_cycles  477253 non-null  int64  
 9   irreversible_damage_index    477253 non-null  float64
 10  thermal_runaway_risk_score   477253 non-null  float64
dtypes: float64(7), int64(3), object(1)
memory usage: 40.1+ MB


,cell_chemistry,cycle,charge_rate_C,discharge_rate_C,cell_temperature_C,internal_resistance_mOhm,capacity_retention_%,cumulative_high_temp_cycles,fast_charge_exposure_cycles,irreversible_damage_index,thermal_runaway_risk_score
0,Li-ion NMC,0,1.311626,2.401429,33.544339,33.923862,99.972475,0,0,0.0,0.02
1,Li-ion NMC,1,1.311626,2.401429,33.359694,33.947258,99.944950,0,0,0.0,0.02
2,Li-ion NMC,2,1.311626,2.401429,33.787926,33.970654,99.917425,0,0,0.0,0.02
3,Li-ion NMC,3,1.311626,2.401429,32.878808,33.994050,99.889901,0,0,0.0,0.02
4,Li-ion NMC,4,1.311626,2.401429,32.150625,34.017447,99.862376,0,0,0.0,0.02


In [20]:
# Unifying labels
cycle_data['cell_chemistry'] = cycle_data['cell_chemistry'].replace({
    'Li-ion NMC': 'NMC'
})

In [22]:
# Filter for relevant types
vehicle_data = vehicle_data[vehicle_data['Battery_Type'].isin(['NMC', 'LFP'])]
cycle_data = cycle_data[cycle_data['cell_chemistry'].isin(['NMC', 'LFP'])]

In [23]:
# Verify the change
print("Unique values now:", cycle_data['cell_chemistry'].unique())
display(cycle_data.head())

Unique values now: ['NMC' 'LFP']


,cell_chemistry,cycle,charge_rate_C,discharge_rate_C,cell_temperature_C,internal_resistance_mOhm,capacity_retention_%,cumulative_high_temp_cycles,fast_charge_exposure_cycles,irreversible_damage_index,thermal_runaway_risk_score
0,NMC,0,1.311626,2.401429,33.544339,33.923862,99.972475,0,0,0.0,0.02
1,NMC,1,1.311626,2.401429,33.359694,33.947258,99.944950,0,0,0.0,0.02
2,NMC,2,1.311626,2.401429,33.787926,33.970654,99.917425,0,0,0.0,0.02
3,NMC,3,1.311626,2.401429,32.878808,33.994050,99.889901,0,0,0.0,0.02
4,NMC,4,1.311626,2.401429,32.150625,34.017447,99.862376,0,0,0.0,0.02
